# EDA

Смотрим что в данных, сколько строк, и как выглядит выручка по месяцам.

In [ ]:
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt

conn = psycopg2.connect(
    host="localhost",
    port=5433,
    database="bdsnowflake",
    user="bds_user",
    password="bds_password",
)

## Сколько строк в staging и в факте

In [ ]:
rows_df = pd.read_sql(
    """
    SELECT 'raw.mock_data_staging' AS table_name, count(*) AS row_count FROM raw.mock_data_staging
    UNION ALL
    SELECT 'dw.fact_sales' AS table_name, count(*) AS row_count FROM dw.fact_sales
    """,
    conn,
)
rows_df

In [ ]:
fk_df = pd.read_sql(
    """
    SELECT
      sum(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) AS null_customer_key,
      sum(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) AS null_product_key,
      sum(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS null_date_key
    FROM dw.fact_sales
    """,
    conn,
)
fk_df

## Выручка по месяцам

In [ ]:
monthly = pd.read_sql(
    """
    SELECT dd.month_number, dd.month_name, sum(fs.sale_total_price) AS revenue
    FROM dw.fact_sales fs
    JOIN dw.dim_date dd ON dd.date_key = fs.date_key
    GROUP BY dd.month_number, dd.month_name
    ORDER BY dd.month_number
    """,
    conn,
)

ax = monthly.plot(x="month_name", y="revenue", kind="bar", legend=False, figsize=(10, 4))
ax.set_title("Выручка по месяцам")
ax.set_xlabel("Месяц")
ax.set_ylabel("Сумма (USD)")
plt.tight_layout()
plt.show()

In [ ]:
conn.close()